# Query Iceberg Orders Table via DuckDB

Reads the `demo.orders` Iceberg table written by the Flink job from MinIO.

**Prerequisites:** the Docker Compose stack must be running (`docker compose up -d`).

In [48]:
import duckdb

con = duckdb.connect()

# Install extensions once, then load them
con.execute("INSTALL httpfs")
con.execute("INSTALL iceberg")
con.execute("LOAD httpfs")
con.execute("LOAD iceberg")

In [49]:
# Point DuckDB at the local MinIO instance
# MinIO is exposed on localhost:9000 from the host machine
con.execute("""
CREATE OR REPLACE SECRET minio (
    TYPE      s3,
    KEY_ID    'minioadmin',
    SECRET    'minioadmin',
    ENDPOINT  'localhost:9000',
    URL_STYLE 'path',
    USE_SSL   false,
    REGION    'us-east-1'
)
""")

In [50]:
# Find the latest metadata file — handles whatever path Nessie/Iceberg chose
files = con.execute("""
    SELECT file
    FROM glob('s3://iceberg-warehouse/**/metadata/*.metadata.json')
    ORDER BY file DESC
    LIMIT 10
""").df()

METADATA_FILE = files["file"].iloc[0]
print(f"Latest metadata: {METADATA_FILE}")
files

Latest metadata: s3://iceberg-warehouse/demo/orders_7ef8ebd3-b08d-4977-8ac9-1312e7593236/metadata/00007-d2243f02-1e76-4055-92a4-b8f12b750c6d.metadata.json


,file
0,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
1,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
2,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
3,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
4,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
5,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
6,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...
7,s3://iceberg-warehouse/demo/orders_7ef8ebd3-b0...


In [51]:
# Select all rows from the Iceberg table
df = con.execute(f"""
    SELECT *
    FROM iceberg_scan('{METADATA_FILE}')
    ORDER BY event_time DESC
""").df()
df

,order_id,customer_id,product_id,quantity,unit_price,status,event_time
0,ord-000468,cust-049,prod-012,7,448.47,CANCELLED,2026-04-23 21:30:15.136
1,ord-000467,cust-035,prod-008,9,346.64,CONFIRMED,2026-04-23 21:30:14.631
2,ord-000466,cust-029,prod-013,4,113.77,PLACED,2026-04-23 21:30:14.127
3,ord-000465,cust-045,prod-005,10,162.95,CANCELLED,2026-04-23 21:30:13.623
4,ord-000464,cust-006,prod-019,2,71.56,SHIPPED,2026-04-23 21:30:13.120
...,...,...,...,...,...,...,...
463,ord-000005,cust-006,prod-016,2,185.95,DELIVERED,2026-04-23 21:26:21.728
464,ord-000004,cust-041,prod-020,9,87.98,CONFIRMED,2026-04-23 21:26:21.225
465,ord-000003,cust-006,prod-019,3,382.64,SHIPPED,2026-04-23 21:26:20.721
466,ord-000002,cust-030,prod-010,2,386.09,PLACED,2026-04-23 21:26:20.214


In [33]:
# Row count and basic stats
con.execute(f"""
    SELECT
        count(*)                             AS total_rows,
        count(DISTINCT customer_id)          AS unique_customers,
        count(DISTINCT product_id)           AS unique_products,
        round(sum(quantity * unit_price), 2) AS total_revenue,
        min(event_time)                      AS earliest_event,
        max(event_time)                      AS latest_event
    FROM iceberg_scan('{METADATA_FILE}')
""").df()

,total_rows,unique_customers,unique_products,total_revenue,earliest_event,latest_event
0,0,0,0,NaN,NaT,NaT


In [ ]:
# Orders by status
con.execute(f"""
    SELECT
        status,
        count(*)                             AS order_count,
        round(sum(quantity * unit_price), 2) AS revenue
    FROM iceberg_scan('{METADATA_FILE}')
    GROUP BY status
    ORDER BY order_count DESC
""").df()

In [ ]:
# Snapshot history
con.execute(f"""
    SELECT *
    FROM iceberg_snapshots('{METADATA_FILE}')
""").df()